In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path
)

documents = [file.parse() for file in reader.read()]
len(documents)

In [ ]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [ ]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

print(documents[0]['filename'])
print(documents[1]['filename'])
print(documents[2]['filename'])

In [ ]:
import json
from evaluation_utils import llm_structured_retry


def generate_ground_truth(doc):

    user_prompt = json.dumps(doc)

    response, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in response.questions:
        results.append({
            "question": q,
            "filename": doc['filename']
        })

    return results, usage



In [ ]:
from evaluation_utils import calc_price, calc_total_price

usages = []
sum_inputs = 0.0
for i in range(3):
    results, usage = generate_ground_truth(documents[i])
    # print(f"{documents[i]['filename']} : {usage['input_tokens']}")
    print(f"{documents[i]['filename']} : {usage}")
    price = calc_price(usage)
    sum_inputs = sum_inputs + price["input_cost"]
    usages.append(usage)

# avg_inputs = usages.sum() / len(usages)

# avg_inputs

# print(sum_inputs)


In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [ ]:
print(chunks[0])

In [ ]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [ ]:
# Build a text Index
from minsearch import Index

tindex = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

tindex.fit(chunks)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
texts = []

for doc in chunks:
    text = doc["content"]
    texts.append(text)

len(texts)

In [ ]:
# Prepare the vectors
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

In [ ]:
import numpy as np

X = np.array(vectors)

In [ ]:
# Build a vector index
from minsearch import VectorSearch

vindex = VectorSearch(
    keyword_fields=["filename"]
)

vindex.fit(X, chunks)

In [ ]:
def text_search(query, num_results=5):

    return tindex.search(
        query,
        num_results=num_results,
    )


In [ ]:
def vector_search(query, num_results=5):
    qvector = model.encode(query)

    return vindex.search(
        qvector,
        num_results=num_results,
    )

In [ ]:
q = ground_truth[0]["question"]

print(q)

In [ ]:
text_results = text_search(q)

text_results

In [ ]:
vector_results = vector_search(q)

vector_results

In [ ]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))
    
    return relevance

In [ ]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)

        relevance_total.append(relevance)

    return relevance_total

In [ ]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt +1
    
    return cnt / len(relevance)

In [ ]:
# Q4 Evaluating text search

relevance_text = compute_relevance_total(ground_truth, text_search)
hit_rate(relevance_text)


In [ ]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [ ]:
# Q5 Evaluating Vector search

relevance_vector = compute_relevance_total(ground_truth, vector_search)
mrr(relevance_vector)

In [ ]:
def rrf(result_lists, k, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
def hybrid_search(query, k=200):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [ ]:
# Q6 Evaluating Hybrid search and tuning

# print(hybrid_search(ground_truth[0]["question"])[0])
# ça retourne une liste avec un scoring fait entre text search et vector

relevance_vector = compute_relevance_total(ground_truth, hybrid_search)
# k = 60
mrr(relevance_vector)